# 01. Первичная разведка набора GeoNames

Цель: не загружая файл целиком в pandas, понять размер, формат, кодировку и структуру исходных данных и составить план очистки.

In [1]:
import sys
from pathlib import Path

# блокнот запускается из notebooks/ - корень проекта нужен для импорта src
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import polars as pl
from IPython.display import Markdown, display

from src import cleaning, config, io_utils

pl.Config.set_tbl_cols(30)
pl.Config.set_tbl_width_chars(160)
pl.Config.set_fmt_str_lengths(60)

polars.config.Config

## Размер, формат, кодировка

Файл читается кусками по 16 МБ только для подсчёта строк. Кодировка проверяется декодированием первых 100 строк в UTF-8.

In [2]:
for path in [config.GEONAMES_CSV, config.HEALTH_CSV, config.WDI_CSV]:
    print(f"{path.name:24s} {io_utils.file_size_mb(path):9.1f} МБ  строк: {io_utils.count_lines(path):,}")

geonames.csv                1404.5 МБ  строк: 11,540,326
health_indicators.csv          1.6 МБ  строк: 5,276
data.csv                      42.9 МБ  строк: 89,011


In [3]:
head = io_utils.peek(config.GEONAMES_CSV, 100)
print(head[0])
print(head[1])
print("запятых в строке:", {line.count(",") for line in head})

geonameid,name,asciiname,alternatenames,latitude,longitude,feature class,feature code,country code,cc2,admin1 code,admin2 code,admin3 code,admin4 code,population,elevation,dem,timezone,modification date
2986043,Pic de Font Blanca,Pic de Font Blanca,Pic de Font Blanca;Pic du Port,42.64991,1.53335,T,PK,AD,,00,,,,0,,2860,Europe/Andorra,2014-11-05
запятых в строке: {18}


Каждая строка содержит ровно 18 запятых, то есть 19 полей; запятых внутри значений нет. Кавычки при этом встречаются внутри названий (`Dar"yoi Amu`) и **не являются** экранированием: если читать файл с обработкой кавычек, разбор ломается и половина строк склеивается. Поэтому чтение выполняется с отключёнными кавычками.

In [4]:
raw = io_utils.scan_geonames().collect()
print(raw.shape)
display(raw.head(5))

(11540325, 19)


geonameid,name,asciiname,alternatenames,latitude,longitude,feature class,feature code,country code,cc2,admin1 code,admin2 code,admin3 code,admin4 code,population,elevation,dem,timezone,modification date
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""2986043""","""Pic de Font Blanca""","""Pic de Font Blanca""","""Pic de Font Blanca;Pic du Port""","""42.64991""","""1.53335""","""T""","""PK""","""AD""",null,"""00""",null,null,null,"""0""",null,"""2860""","""Europe/Andorra""","""2014-11-05"""
"""2994701""","""Roc Mélé""","""Roc Mele""","""Roc Mele;Roc Meler;Roc Mélé""","""42.58765""","""1.74028""","""T""","""MT""","""AD""","""AD;FR""","""00""",null,null,null,"""0""",null,"""2803""","""Europe/Andorra""","""2014-11-05"""
"""3007683""","""Pic des Langounelles""","""Pic des Langounelles""","""Pic des Langounelles""","""42.61203""","""1.47364""","""T""","""PK""","""AD""","""AD;FR""","""00""",null,null,null,"""0""",null,"""2685""","""Europe/Andorra""","""2014-11-05"""
"""3017832""","""Pic de les Abelletes""","""Pic de les Abelletes""","""Pic de la Font-Negre;Pic de la Font-Nègre;Pic de les Abellet…","""42.52535""","""1.73343""","""T""","""PK""","""AD""","""FR""","""A9""","""66""","""663""","""66146""","""0""",null,"""2411""","""Europe/Andorra""","""2014-11-05"""
"""3017833""","""Estany de les Abelletes""","""Estany de les Abelletes""","""Estany de les Abelletes;Etang de Font-Negre;Étang de Font-Nè…","""42.52915""","""1.73362""","""H""","""LK""","""AD""","""FR""","""A9""",null,null,null,"""0""",null,"""2260""","""Europe/Andorra""","""2014-11-05"""


## Типы, уникальные значения и пропуски

Все столбцы прочитаны как строки - так видно исходное содержимое без угадывания типов.

In [5]:
overview = pl.DataFrame({
    "column": raw.columns,
    "n_unique": [raw[c].n_unique() for c in raw.columns],
    "missing_share": [round(raw[c].null_count() / raw.height, 4) for c in raw.columns],
    "example": [str(raw[c].drop_nulls().head(1).to_list()) for c in raw.columns],
})
display(overview)

column,n_unique,missing_share,example
str,i64,f64,str
"""geonameid""",11540325,0.0,"""['2986043']"""
"""name""",7234262,0.0,"""['Pic de Font Blanca']"""
"""asciiname""",7111008,0.0,"""['Pic de Font Blanca']"""
"""alternatenames""",4225199,0.516,"""['Pic de Font Blanca;Pic du Port']"""
"""latitude""",4826591,0.0,"""['42.64991']"""
…,…,…,…
"""population""",53068,0.0,"""['0']"""
"""elevation""",5944,0.7963,"""['2761']"""
"""dem""",6716,0.0,"""['2860']"""


## Скрытые пропуски и подозрительные значения

In [6]:
checks = raw.select(
    (pl.col("population") == "0").mean().alias("population = 0"),
    (pl.col("dem") == "-9999").mean().alias("dem = -9999"),
    (pl.col("country code") == "NA").sum().alias("country code = NA (Намибия)"),
    pl.col("alternatenames").str.contains('"').sum().alias('кавычки в alternatenames'),
    (pl.col("name") != pl.col("name").str.strip_chars()).sum().alias("краевые пробелы в name"),
)
display(checks)

population = 0,dem = -9999,country code = NA (Намибия),кавычки в alternatenames,краевые пробелы в name
f64,f64,u32,u32,u32
0.956897,0.022318,7946,3807,7


In [7]:
display(cleaning.missing_by_group(
    raw.with_columns(pl.when(pl.col("population") != "0").then(pl.col("population")).alias("population")),
    "population", "feature class"))

feature class,rows,population_missing_share
str,u32,f64
"""P""",4577639,0.9197
"""S""",2340891,0.9989
"""H""",2186251,1.0
"""T""",1557088,0.9998
"""L""",400030,0.9992
"""A""",371320,0.6586
"""V""",45113,0.9998
"""R""",42400,0.9996
"""U""",14519,0.9999


Население не указано почти у всех объектов, кроме населённых пунктов (P) и административных единиц (A). Пропуски сосредоточены в классах, где население не имеет смысла, поэтому строки не удаляются, а 0 заменяется на пропуск.

## Вспомогательные источники

In [8]:
health = io_utils.read_health()
print(health.shape)
display(health.select("country_code", "country_name", "region", "income_level", "year").head(3))
print("регионы:", health["region"].unique().to_list())

(5275, 48)


country_code,country_name,region,income_level,year
str,str,str,str,i64
"""ABW""","""Aruba""","""Latin America & Caribbean ""","""High income""",2000
"""ABW""","""Aruba""","""Latin America & Caribbean ""","""High income""",2001
"""ABW""","""Aruba""","""Latin America & Caribbean ""","""High income""",2002


регионы: ['North America', 'Sub-Saharan Africa ', 'Middle East, North Africa, Afghanistan & Pakistan', 'East Asia & Pacific', 'Latin America & Caribbean ', 'South Asia', 'Europe & Central Asia']


In [9]:
wdi = io_utils.read_wdi()
print(wdi.shape)
print(wdi.columns[:5], "...", wdi.columns[-3:])

(89010, 61)
['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960'] ... ['2014', '2015', '']


## Обнаруженные дефекты - план очистки

1. Нет экранирования, символ `"` внутри значений - читать с `quote_char=None` / `QUOTE_NONE`.
2. Код страны `NA` (Намибия) pandas по умолчанию превращает в пропуск - `keep_default_na=False`.
3. Без схемы pandas читает admin-коды как числа и теряет ведущие нули (`09` → `9`).
4. Скрытые пропуски: `population = 0`, `dem = -9999`.
5. Списки в одной ячейке (`alternatenames`, `cc2`) - нарушение 1НФ.
6. Повторы одной сущности под разными `geonameid` (совпадают имя, код, страна, координаты).
7. Краевые и двойные пробелы в названиях, пустые элементы в списках.
8. Выбросы: население в миллиарды у гор и рек, расхождение `elevation` и `dem` больше 1 км.
9. Часть `elevation` похожа на футы (отношение к `dem` около 3,28).
10. Коды стран в разных системах: ISO2 в GeoNames, ISO3 в health и WDI; устаревшие коды CS, YU, AN, XK.
11. `region` в health с краевыми пробелами; столбец `obesity_pct` пуст целиком.
12. WDI в широком формате (год - столбец), лишний пустой столбец, агрегаты регионов вперемешку со странами.